# EJERCICIO SENCILLO DE ANALISIS DE SENTIMIENTOS USANDO TRANSFORMERS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

# PASO 1 - CREAMOS UN DATASET DE PRUEBA

In [2]:
# Datos simples
frases = [
    "me encanta este producto",
    "odio este artículo",
    "muy bueno y útil",
    "es terrible",
    "perfecto para mí",
    "es muy malo"
]

# Etiquetas: 1 = positivo, 0 = negativo
labels = [1, 0, 1, 0, 1, 0]

# PASO 2 - PROCESAMIENTO DE LOS DATOS

In [3]:
# Crear vocabulario simple
vocab = {"<PAD>": 0}
for frase in frases:
    for palabra in frase.split():
        if palabra not in vocab:
            vocab[palabra] = len(vocab)

In [4]:
# Convertir frases a secuencias de índices
max_len = 6  # Longitud máxima de las frases

def tokenize(frase):
    tokens = [vocab.get(palabra, 0) for palabra in frase.split()]
    # Rellenar hasta max_len
    return tokens + [0] * (max_len - len(tokens))

In [5]:
X = torch.tensor([tokenize(frase) for frase in frases])
y = torch.tensor(labels)

# CREAMOS NUESTRO MODELO TRANSFORMER

In [6]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        embedded = self.embedding(x).permute(1, 0, 2)  # seq_len, batch_size, embed_dim
        encoded = self.transformer_encoder(embedded)
        output = encoded.mean(dim=0)  # Promediar tokens
        return self.fc(output)

# PARAMETROS

In [7]:
embed_dim = 32
num_heads = 4
num_classes = 2

In [8]:
model = TransformerClassifier(vocab_size=len(vocab), embed_dim=embed_dim, num_heads=num_heads, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

/tmp/ipykernel_4600/1204796438.py:6: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)


# ENTRENAMIENTO

In [9]:
epochs = 300

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 50, Loss: 0.0036
Epoch 100, Loss: 0.0019
Epoch 150, Loss: 0.0013
Epoch 200, Loss: 0.0010
Epoch 250, Loss: 0.0007
Epoch 300, Loss: 0.0005


# PROBAMOS EL MODELO

In [10]:
frase_nueva = "muy bueno y útil"
x_nueva = torch.tensor([tokenize(frase_nueva)])

with torch.no_grad():
    salida = model(x_nueva)
    prediccion = torch.argmax(salida, dim=1).item()

print("Predicción:", "Positiva" if prediccion == 1 else "Negativa")

Predicción: Positiva
